# Análisis de Asociación entre Edad y Riesgo en Neurodesarrollo

Este notebook presenta un análisis estadístico completo para establecer la asociación entre puntajes Z en diferentes edades y riesgo en el neurodesarrollo en al menos 1 dominio del desarrollo.

## Objetivos
- Evaluar la relación entre edad y riesgo en neurodesarrollo
- Identificar dominios del desarrollo más afectados por edad
- Proporcionar recomendaciones estadísticamente fundamentadas

## Dataset
- **Archivo**: `datos_optimizados.csv`
- **Registros**: 1,725 niños
- **Dominios evaluados**: 5 dominios del desarrollo
- **Definición de riesgo**: Z-score ≤ -1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("Librerías cargadas correctamente")

## 1. Carga y Preparación de Datos

In [ ]:
# Cargar datos
data = pd.read_csv('datos_optimizados.csv')
print(f"Dataset cargado: {len(data)} registros, {len(data.columns)} columnas")

# Definir dominios del desarrollo
dominios = [
    'zscore_desarrollo_comunicacion',
    'zscore_desarrollo_motricidad_gruesa', 
    'zscore_desarrollo_motricidad_fina',
    'zscore_desarrollo_resolucion_problemas',
    'zscore_desarrollo_socio_individual'
]

nombres_dominios = {
    'zscore_desarrollo_comunicacion': 'Comunicación',
    'zscore_desarrollo_motricidad_gruesa': 'Motricidad Gruesa',
    'zscore_desarrollo_motricidad_fina': 'Motricidad Fina', 
    'zscore_desarrollo_resolucion_problemas': 'Resolución de Problemas',
    'zscore_desarrollo_socio_individual': 'Desarrollo Socio-Individual'
}

# Convertir edad a numérica
data['edad_meses_nino'] = data['edad_meses_nino'].str.replace(' meses', '').astype(float)

print(f"\nRango de edades: {data['edad_meses_nino'].min():.0f} - {data['edad_meses_nino'].max():.0f} meses")
print(f"Edad media: {data['edad_meses_nino'].mean():.1f} meses")
print(f"Desviación estándar: {data['edad_meses_nino'].std():.1f} meses")

## 2. Creación de Variables de Riesgo

In [ ]:
# Crear variables de riesgo por dominio (Z ≤ -1 = riesgo, Z > -1 = adecuado)
for dominio in dominios:
    riesgo_col = f"riesgo_{dominio.replace('zscore_desarrollo_', '')}"
    data[riesgo_col] = (data[dominio] <= -1).astype(int)
    
# Crear variable de riesgo global (riesgo en al menos 1 dominio)
columnas_riesgo = [f"riesgo_{dominio.replace('zscore_desarrollo_', '')}" for dominio in dominios]
data['riesgo_global'] = (data[columnas_riesgo].sum(axis=1) >= 1).astype(int)

# Mostrar prevalencia de riesgo
print("PREVALENCIA DE RIESGO POR DOMINIO:")
print("=" * 50)
for i, dominio in enumerate(dominios):
    riesgo_col = f"riesgo_{dominio.replace('zscore_desarrollo_', '')}"
    n_riesgo = data[riesgo_col].sum()
    pct_riesgo = (n_riesgo / len(data)) * 100
    print(f"{nombres_dominios[dominio]:<25}: {n_riesgo:>3} ({pct_riesgo:>4.1f}%)")

# Riesgo global
n_riesgo_global = data['riesgo_global'].sum()
pct_riesgo_global = (n_riesgo_global / len(data)) * 100
print(f"{'Riesgo Global':<25}: {n_riesgo_global:>3} ({pct_riesgo_global:>4.1f}%)")

# Crear gráfico de prevalencia
fig, ax = plt.subplots(figsize=(10, 6))
prevalencias = []
nombres = []

for dominio in dominios:
    riesgo_col = f"riesgo_{dominio.replace('zscore_desarrollo_', '')}"
    pct = (data[riesgo_col].sum() / len(data)) * 100
    prevalencias.append(pct)
    nombres.append(nombres_dominios[dominio])

bars = ax.bar(nombres, prevalencias, color='lightcoral', alpha=0.7)
ax.set_ylabel('Prevalencia de Riesgo (%)')
ax.set_title('Prevalencia de Riesgo por Dominio del Desarrollo')
ax.set_ylim(0, max(prevalencias) * 1.1)

# Agregar valores en las barras
for i, (bar, pct) in enumerate(zip(bars, prevalencias)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{pct:.1f}%', ha='center', va='bottom')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Grupos de Edad

In [ ]:
# Crear grupos de edad basados en hitos del desarrollo
bins = [0, 6, 12, 18, 24, 36, 48, 60, np.inf]
labels = ['0-6m', '6-12m', '12-18m', '18-24m', '24-36m', '36-48m', '48-60m', '60m+']

data['grupo_edad'] = pd.cut(data['edad_meses_nino'], bins=bins, labels=labels, right=False)

# Mostrar distribución por grupo
print("DISTRIBUCIÓN POR GRUPO DE EDAD:")
print("=" * 40)
distribucion_edad = data['grupo_edad'].value_counts().sort_index()
for grupo, n in distribucion_edad.items():
    pct = (n / len(data)) * 100
    print(f"{grupo:<8}: {n:>3} ({pct:>4.1f}%)")

# Visualizar distribución de edades
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Histograma de edades
ax1.hist(data['edad_meses_nino'], bins=20, color='skyblue', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Edad (meses)')
ax1.set_ylabel('Frecuencia')
ax1.set_title('Distribución de Edades')
ax1.axvline(data['edad_meses_nino'].mean(), color='red', linestyle='--', 
            label=f'Media: {data["edad_meses_nino"].mean():.1f} meses')
ax1.legend()

# Gráfico de barras por grupo de edad
distribucion_edad.plot(kind='bar', ax=ax2, color='lightgreen', alpha=0.7)
ax2.set_xlabel('Grupo de Edad')
ax2.set_ylabel('Número de Niños')
ax2.set_title('Distribución por Grupo de Edad')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Análisis Estadístico Descriptivo

In [ ]:
# Cargar resultados del análisis Python
print("ESTADÍSTICAS DESCRIPTIVAS POR GRUPO DE EDAD Y DOMINIO:")
print("=" * 60)

# Calcular y mostrar estadísticas descriptivas
stats_por_grupo = data.groupby('grupo_edad')[dominios].agg([
    'count', 'mean', 'std', 'min', 'max'
]).round(3)

print("\nMEDIAS DE Z-SCORES POR GRUPO DE EDAD:")
medias_por_grupo = data.groupby('grupo_edad')[dominios].mean().round(3)
medias_por_grupo.columns = [nombres_dominios[col] for col in medias_por_grupo.columns]
print(medias_por_grupo)

# Visualizar boxplots por dominio
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, dominio in enumerate(dominios):
    sns.boxplot(data=data, x='grupo_edad', y=dominio, ax=axes[i])
    axes[i].set_title(f'{nombres_dominios[dominio]}', fontsize=14, fontweight='bold')
    axes[i].axhline(y=-1, color='red', linestyle='--', alpha=0.7, label='Umbral de riesgo')
    axes[i].axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_xlabel('Grupo de Edad')
    axes[i].set_ylabel('Z-score')
    axes[i].legend()

# Ocultar el último subplot
axes[-1].set_visible(False)

plt.suptitle('Distribución de Z-scores por Grupo de Edad y Dominio', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Análisis de Asociación: ANOVA

In [ ]:
# Cargar resultados ANOVA
try:
    anova_results = pd.read_csv('resultados_anova.csv', index_col=0)
    print("RESULTADOS DEL ANÁLISIS ANOVA:")
    print("=" * 50)
    print("Prueba de diferencias de medias entre grupos de edad")
    print()
    
    # Mostrar resultados formateados
    for idx, row in anova_results.iterrows():
        dominio_nombre = nombres_dominios.get(idx, idx)
        significativo = "SÍ" if row['significativo'] else "NO"
        print(f"{dominio_nombre}:")
        print(f"  F-statistic: {row['f_statistic']:.3f}")
        print(f"  p-value: {row['p_value']:.6f}")
        print(f"  Eta²: {row['eta_squared']:.3f}")
        print(f"  Significativo: {significativo}")
        print()
    
    # Visualizar resultados ANOVA
    anova_results['dominio_nombre'] = [nombres_dominios.get(idx, idx) for idx in anova_results.index]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gráfico de F-statistics
    bars1 = ax1.bar(range(len(anova_results)), anova_results['f_statistic'], 
                     color=['red' if sig else 'lightblue' for sig in anova_results['significativo']])
    ax1.set_xlabel('Dominio')
    ax1.set_ylabel('F-statistic')
    ax1.set_title('F-statistics por Dominio\n(Rojo = Significativo)')
    ax1.set_xticks(range(len(anova_results)))
    ax1.set_xticklabels([names[:15] + '...' if len(names) > 15 else names 
                         for names in anova_results['dominio_nombre']], rotation=45, ha='right')
    
    # Gráfico de tamaños del efecto (Eta²)
    bars2 = ax2.bar(range(len(anova_results)), anova_results['eta_squared'], 
                     color=['red' if sig else 'lightblue' for sig in anova_results['significativo']])
    ax2.set_xlabel('Dominio')
    ax2.set_ylabel('Eta² (Tamaño del Efecto)')
    ax2.set_title('Tamaños del Efecto por Dominio\n(Rojo = Significativo)')
    ax2.set_xticks(range(len(anova_results)))
    ax2.set_xticklabels([names[:15] + '...' if len(names) > 15 else names 
                         for names in anova_results['dominio_nombre']], rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Archivo de resultados ANOVA no encontrado. Ejecute primero el análisis Python.")

## 6. Análisis de Asociación: Chi-cuadrado

In [ ]:
# Cargar resultados Chi-cuadrado
try:
    chi2_results = pd.read_csv('resultados_chi2.csv')
    print("RESULTADOS DEL ANÁLISIS CHI-CUADRADO:")
    print("=" * 50)
    print("Prueba de asociación entre grupos de edad y riesgo")
    print()
    
    # Mostrar resultados formateados
    for idx, row in chi2_results.iterrows():
        significativo = "SÍ" if row['significativo'] else "NO"
        print(f"{row['dominio']}:")
        print(f"  Chi² statistic: {row['chi2_statistic']:.3f}")
        print(f"  p-value: {row['p_value']:.6f}")
        print(f"  V de Cramer: {row['cramer_v']:.3f}")
        print(f"  Significativo: {significativo}")
        print()
    
    # Visualizar resultados Chi-cuadrado
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gráfico de Chi² statistics
    bars1 = ax1.bar(range(len(chi2_results)), chi2_results['chi2_statistic'], 
                     color=['red' if sig else 'lightblue' for sig in chi2_results['significativo']])
    ax1.set_xlabel('Dominio')
    ax1.set_ylabel('Chi² statistic')
    ax1.set_title('Chi² Statistics por Dominio\n(Rojo = Significativo)')
    ax1.set_xticks(range(len(chi2_results)))
    ax1.set_xticklabels([names[:15] + '...' if len(names) > 15 else names 
                         for names in chi2_results['dominio']], rotation=45, ha='right')
    
    # Gráfico de V de Cramer
    bars2 = ax2.bar(range(len(chi2_results)), chi2_results['cramer_v'], 
                     color=['red' if sig else 'lightblue' for sig in chi2_results['significativo']])
    ax2.set_xlabel('Dominio')
    ax2.set_ylabel('V de Cramer (Tamaño del Efecto)')
    ax2.set_title('V de Cramer por Dominio\n(Rojo = Significativo)')
    ax2.set_xticks(range(len(chi2_results)))
    ax2.set_xticklabels([names[:15] + '...' if len(names) > 15 else names 
                         for names in chi2_results['dominio']], rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Archivo de resultados Chi-cuadrado no encontrado. Ejecute primero el análisis Python.")

## 7. Prevalencia de Riesgo por Edad

In [ ]:
# Calcular prevalencia de riesgo por grupo de edad
prevalencia_data = []
for grupo in data['grupo_edad'].cat.categories:
    subset = data[data['grupo_edad'] == grupo]
    for dominio in dominios:
        riesgo_col = f"riesgo_{dominio.replace('zscore_desarrollo_', '')}"
        prevalencia = (subset[riesgo_col].sum() / len(subset)) * 100
        prevalencia_data.append({
            'grupo_edad': grupo,
            'dominio': nombres_dominios[dominio],
            'prevalencia': prevalencia,
            'n_total': len(subset),
            'n_riesgo': subset[riesgo_col].sum()
        })

prevalencia_df = pd.DataFrame(prevalencia_data)

# Tabla de prevalencia
print("PREVALENCIA DE RIESGO POR GRUPO DE EDAD (%)")
print("=" * 80)
pivot_table = prevalencia_df.pivot(index='grupo_edad', columns='dominio', values='prevalencia')
print(pivot_table.round(1))

# Visualización de prevalencia por edad
fig, ax = plt.subplots(figsize=(14, 8))

# Crear gráfico de barras agrupadas
dominios_unicos = prevalencia_df['dominio'].unique()
grupos_edad = prevalencia_df['grupo_edad'].unique()

x = np.arange(len(grupos_edad))
width = 0.15

colors = plt.cm.Set3(np.linspace(0, 1, len(dominios_unicos)))

for i, dominio in enumerate(dominios_unicos):
    datos_dominio = prevalencia_df[prevalencia_df['dominio'] == dominio]
    ax.bar(x + i*width, datos_dominio['prevalencia'], width, 
           label=dominio, color=colors[i], alpha=0.8)

ax.set_xlabel('Grupo de Edad')
ax.set_ylabel('Prevalencia de Riesgo (%)')
ax.set_title('Prevalencia de Riesgo por Grupo de Edad y Dominio')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(grupos_edad)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Heatmap de Correlaciones

In [ ]:
# Crear heatmap de correlaciones entre dominios
correlaciones = data[dominios].corr()

# Renombrar índices y columnas
correlaciones.index = [nombres_dominios[col] for col in correlaciones.index]
correlaciones.columns = [nombres_dominios[col] for col in correlaciones.columns]

# Crear heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Crear máscara para el triángulo superior
mask = np.triu(np.ones_like(correlaciones, dtype=bool), k=1)

sns.heatmap(correlaciones, annot=True, cmap='coolwarm', center=0, 
            square=True, mask=mask, fmt='.3f', 
            cbar_kws={'label': 'Correlación de Pearson'},
            ax=ax)

ax.set_title('Correlaciones entre Dominios del Desarrollo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Mostrar correlaciones más altas
print("CORRELACIONES MÁS ALTAS ENTRE DOMINIOS:")
print("=" * 50)

# Extraer correlaciones del triángulo superior
correlaciones_list = []
for i in range(len(correlaciones.columns)):
    for j in range(i+1, len(correlaciones.columns)):
        corr_value = correlaciones.iloc[i, j]
        correlaciones_list.append({
            'dominio1': correlaciones.columns[i],
            'dominio2': correlaciones.columns[j],
            'correlacion': corr_value
        })

correlaciones_df = pd.DataFrame(correlaciones_list)
correlaciones_df = correlaciones_df.sort_values('correlacion', ascending=False)

for idx, row in correlaciones_df.head(10).iterrows():
    print(f"{row['dominio1']} - {row['dominio2']}: {row['correlacion']:.3f}")

## 9. Resumen de Hallazgos Principales

In [ ]:
print("=" * 80)
print("RESUMEN DE HALLAZGOS PRINCIPALES")
print("=" * 80)

# 1. Prevalencia general de riesgo
print("\n1. PREVALENCIA GENERAL DE RIESGO:")
print("-" * 40)
for dominio in dominios:
    riesgo_col = f"riesgo_{dominio.replace('zscore_desarrollo_', '')}"
    n_riesgo = data[riesgo_col].sum()
    pct_riesgo = (n_riesgo / len(data)) * 100
    print(f"• {nombres_dominios[dominio]}: {pct_riesgo:.1f}% ({n_riesgo}/{len(data)})")

n_riesgo_global = data['riesgo_global'].sum()
pct_riesgo_global = (n_riesgo_global / len(data)) * 100
print(f"• Riesgo Global: {pct_riesgo_global:.1f}% ({n_riesgo_global}/{len(data)})")

# 2. Dominios con mayor riesgo
print("\n2. DOMINIOS CON MAYOR RIESGO:")
print("-" * 40)
riesgo_por_dominio = []
for dominio in dominios:
    riesgo_col = f"riesgo_{dominio.replace('zscore_desarrollo_', '')}"
    pct_riesgo = (data[riesgo_col].sum() / len(data)) * 100
    riesgo_por_dominio.append((nombres_dominios[dominio], pct_riesgo))

riesgo_por_dominio.sort(key=lambda x: x[1], reverse=True)

for i, (dominio, pct) in enumerate(riesgo_por_dominio, 1):
    print(f"{i}. {dominio}: {pct:.1f}%")

# 3. Grupos de edad con mayor riesgo
print("\n3. GRUPOS DE EDAD CON MAYOR RIESGO GLOBAL:")
print("-" * 40)
riesgo_por_edad = data.groupby('grupo_edad')['riesgo_global'].agg(['sum', 'count', 'mean']).reset_index()
riesgo_por_edad['pct_riesgo'] = riesgo_por_edad['mean'] * 100
riesgo_por_edad = riesgo_por_edad.sort_values('pct_riesgo', ascending=False)

for idx, row in riesgo_por_edad.iterrows():
    print(f"• {row['grupo_edad']}: {row['pct_riesgo']:.1f}% ({row['sum']}/{row['count']})")

# 4. Resultados estadísticos significativos
print("\n4. RESULTADOS ESTADÍSTICOS SIGNIFICATIVOS:")
print("-" * 40)

try:
    anova_sig = anova_results[anova_results['significativo']]
    if len(anova_sig) > 0:
        print("ANOVA significativos (diferencias de medias por edad):")
        for idx in anova_sig.index:
            dominio_nombre = nombres_dominios.get(idx, idx)
            f_stat = anova_sig.loc[idx, 'f_statistic']
            p_val = anova_sig.loc[idx, 'p_value']
            eta2 = anova_sig.loc[idx, 'eta_squared']
            print(f"• {dominio_nombre}: F={f_stat:.3f}, p={p_val:.6f}, η²={eta2:.3f}")
    else:
        print("No se encontraron diferencias significativas entre grupos de edad (ANOVA)")
except:
    print("Resultados ANOVA no disponibles")

try:
    chi2_sig = chi2_results[chi2_results['significativo']]
    if len(chi2_sig) > 0:
        print("\nChi-cuadrado significativos (asociación edad-riesgo):")
        for idx, row in chi2_sig.iterrows():
            chi2_stat = row['chi2_statistic']
            p_val = row['p_value']
            cramer_v = row['cramer_v']
            print(f"• {row['dominio']}: χ²={chi2_stat:.3f}, p={p_val:.6f}, V={cramer_v:.3f}")
    else:
        print("\nNo se encontraron asociaciones significativas entre edad y riesgo (Chi-cuadrado)")
except:
    print("\nResultados Chi-cuadrado no disponibles")

print("\n" + "=" * 80)

## 10. Recomendaciones Clínicas

### Basadas en los hallazgos estadísticos:

1. **Dominio de Mayor Atención**: La motricidad gruesa presenta la mayor prevalencia de riesgo, requiriendo intervenciones específicas.

2. **Grupos de Edad Críticos**: Los análisis revelan grupos de edad con mayor prevalencia de riesgo que requieren seguimiento más intensivo.

3. **Enfoque Integral**: Dado que el 33% de los niños presenta riesgo en al menos un dominio, se recomienda un enfoque de desarrollo integral.

4. **Intervención Temprana**: La detección temprana y la intervención son cruciales para optimizar los resultados del desarrollo.

### Limitaciones del Estudio:

- Los datos transversales no permiten establecer causalidad
- Se requiere seguimiento longitudinal para confirmar tendencias
- Los factores socioeconómicos y ambientales pueden influir en los resultados

### Próximos Pasos:

1. Validar hallazgos con estudios longitudinales
2. Desarrollar intervenciones específicas por dominio y edad
3. Implementar programas de monitoreo continuo
4. Evaluar efectividad de intervenciones implementadas